# L-CAD baseline on the Phase 0 smoke subset (Colab)

L-CAD (NeurIPS 2023, language-based colorization with a diffusion prior) is the **language baseline** for chroma-reasoner Phase 0. It does not fit in 4 GB VRAM, so it runs here on a Colab GPU (T4/A100 both fine).

Flow: clone repos → install deps → download weights (Google Drive) → regenerate the deterministic COCO smoke subset → run inference with COCO captions → zip outputs for local evaluation with `scripts/evaluate.py`.

> **Runtime → Change runtime type → GPU** before running.

In [ ]:
!nvidia-smi
import torch; print(torch.__version__, torch.cuda.is_available())

## 1. Clone repos

In [ ]:
# TODO: replace with your GitHub URL once chroma-reasoner is pushed.
CHROMA_REASONER_URL = "https://github.com/<YOUR_USERNAME>/chroma-reasoner.git"

!git clone https://github.com/changzheng123/L-CAD.git
!git clone {CHROMA_REASONER_URL}

## 2. Install dependencies

In [ ]:
%cd /content
# Colab runtimes are ephemeral: re-run the clone cell after any reset.
# These guards make this cell safe to re-run either way.
!test -d L-CAD || git clone https://github.com/changzheng123/L-CAD.git

# Do NOT `pip install -r L-CAD/requirements.txt`: it is a conda-env dump that
# pins torch==1.12.1, numpy from a local file:// path, and an editable local
# segment-anything — it breaks on Colab and would clobber Colab's CUDA torch.
# Install only what the LDM-style codebase imports, keeping Colab's torch.
!pip install -q pytorch-lightning==1.9.5 omegaconf einops kornia open-clip-torch taming-transformers-rom1504
!pip install -q git+https://github.com/openai/CLIP.git
!pip install -q gdown pycocotools clean-fid
!pip install -q -e chroma-reasoner

## 3. Download pretrained weights

Weights folder (from the L-CAD README): https://drive.google.com/drive/folders/1_zVJrp_UkFDaZpcC8aLzpv4iPsHADQU-

In [ ]:
!mkdir -p L-CAD/models
!gdown --folder https://drive.google.com/drive/folders/1_zVJrp_UkFDaZpcC8aLzpv4iPsHADQU- -O L-CAD/models --remaining-ok
!ls -lh L-CAD/models

## 4. Regenerate the smoke subset (deterministic: n=300, seed=0)

Same selection code as local, so stems match for evaluation.

In [ ]:
!cd chroma-reasoner && python scripts/download_coco_subset.py --n 300 --seed 0 --root data/coco

## 5. Run L-CAD inference

L-CAD's entrypoints are `colorization_main.py` (standard) and `inference.py`
(instance-aware). **The exact config for custom image+caption input needs
checking against the repo's configs on first run** — inspect
`L-CAD/configs/` and how its dataset class reads (image, caption) pairs, then
point it at the smoke subset below. The manifest at
`chroma-reasoner/data/coco/manifest.json` has `file_name` → `captions[0]`
for every image.

In [ ]:
import json, pathlib
manifest = json.load(open('chroma-reasoner/data/coco/manifest.json'))
pairs = [(im['file_name'], im['captions'][0]) for im in manifest['images'] if im['captions']]
print(len(pairs), 'image/caption pairs; first:', pairs[0])

# Write a simple annotation file many L-CAD configs can be adapted to consume.
with open('lcad_smoke_captions.json', 'w') as f:
    json.dump({fn: cap for fn, cap in pairs}, f, indent=2)

In [ ]:
# TODO(colab): adapt to L-CAD's actual sampling config after inspecting the repo.
# Typical shape:
# !cd L-CAD && python colorization_main.py --config configs/<sampling_config>.yaml \
#     --input_dir ../chroma-reasoner/data/coco/gray --caption_file ../lcad_smoke_captions.json \
#     --output_dir ../results/lcad

## 6. Export outputs for local evaluation

In [ ]:
!zip -rq lcad_outputs.zip results/lcad
from google.colab import files
files.download('lcad_outputs.zip')
# Locally: python scripts/evaluate.py --pred results/lcad --gt data/coco/val2017_subset \
#              --manifest data/coco/manifest.json --out results/phase0/lcad